# Rust 107: I/O and Files — Practice

Companion exercises for `07_io-and-files_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and there is one exercise for each of its demonstration cells, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the parts marked `TODO`; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. The World Outside the Program: Errors as Values](#1-the-world-outside-the-program-errors-as-values)
- [2. stdout and stderr](#2-stdout-and-stderr)
- [3. Paths: `Path`, `PathBuf` and Platform Truth](#3-paths-path-pathbuf-and-platform-truth)
- [4. Reading and Writing Whole Files](#4-reading-and-writing-whole-files)
- [5. File Handles, `OpenOptions` and Append](#5-file-handles-openoptions-and-append)
- [6. Buffered I/O: `BufReader` and `BufWriter`](#6-buffered-io-bufreader-and-bufwriter)
- [7. Working with Directories](#7-working-with-directories)
- [8. Command-Line Arguments](#8-command-line-arguments)
- [9. Environment Variables](#9-environment-variables)
- [10. Running Other Programs](#10-running-other-programs)
- [11. Reading the Compiler: Panics and E-Codes](#11-reading-the-compiler-panics-and-e-codes)
- [12. Performance Notes for Low-Latency Code](#12-performance-notes-for-low-latency-code)
- [13. Mental Model: The Program as a Guest](#13-mental-model-the-program-as-a-guest)
- [16. Review: Mixed Problems](#16-review-mixed-problems)

§1–§13 each have one exercise per demonstration cell of the example notebook; §16 mixes them. When a check fails, the fastest route back is §15 of the example notebook, which is the one-page cheat sheet.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from one cell is visible in every cell below it.
3. **Items are order-independent.** A `fn`, a `const` or a `use` line may be used above the cell that defines it.

A fourth consequence bites harder in this chapter than anywhere else in the track:

> **Item names must be unique across the whole notebook.** Every cell shares one namespace, so two functions with the same name are a redefinition error, even in different sections. If the compiler says a name "is defined multiple times", it is pointing at an *earlier* cell. Rename the new one.

The same one-namespace rule applies to `use` lines: this notebook imports each path **once**, at its first exercise (`std::fs` in 0.1, `std::io` in 1.1, `std::io::Write` in 5.1, the `BufRead` family in 6.1–6.2, `Instant` in 12.1). Do not re-import per cell.

### The stub convention

Each exercise cell declares a signature, a doc comment stating the contract, and a `// --- check (do not modify) ---` block of `assert_eq!`/`assert!` calls. The body you must fill in is `todo!()`, which compiles but **panics** when executed:

```text
thread '<main>' panicked at ... not yet implemented
```

Replace the `todo!()` and re-run the cell. An `Exercise N.M passed` line means you are done.

### Two notes specific to I/O

- **The sandbox is `scratch07p/`.** Check blocks create their own fixture files under it, so every cell is self-sufficient and re-runnable; the last review exercise deletes the whole sandbox. The kernel's working directory is wherever Jupyter was started — paths in this notebook are relative to it.
- **The kernel host owns `args()` and the environment.** `std::env::args()` yields the *kernel's* arguments, not your program's, so the argv exercises pass argument vectors as parameters instead; the environment exercises use a key unique to this notebook. Remember `env::set_var`/`remove_var` are `unsafe` in edition 2024.

### Expect warnings until you finish

Every unsolved stub whose parameters are untouched emits `unused variable`, and until the exercise that uses them is solved, the notebook-wide imports emit `unused import` too (`std::fs` in 0.1, `std::io` in 1.1, `Write` in 5.1, `BufRead`/`BufReader` in 6.1, `Instant` in 12.1). So a fresh notebook shows a long list of both. That is normal and self-cleaning: each warning disappears when you implement the body that needs it. If a diagnostic appears that is not in one of those two categories, it is yours.

**See**: §0 and §11 of the example notebook.


### Exercise 0.1 — The kernel can touch the disk

**Goal**: prove the sandbox directory can be created and seen from a cell.

**Given**: nothing; this is the smoke test. `std::fs` is imported here once for the whole notebook.

**Objectives**

- implement `sandbox_ready` so it creates the directory (idempotently) and reports whether it exists
- keep the signature unchanged

**See**: §0.1 and §3.3 of the example notebook.

**Hint**: the example notebook's cells all begin with the same two-call ritual: `create_dir_all`, then a check. `fs::exists` returns `io::Result<bool>` — and it does *not* error on a missing path.


In [ ]:
use std::fs;

/// Create `path` (and parents) if needed, then report whether it exists.
///
/// TODO: replace `todo!()` with the create-then-check ritual.
fn sandbox_ready(path: &str) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(sandbox_ready("scratch07p"), true);
assert_eq!(sandbox_ready("scratch07p/nested/deeper"), true);
assert_eq!(sandbox_ready("scratch07p/nested/deeper"), true); // idempotent
println!("Exercise 0.1 passed");


### Exercise 1.1 — Classify a read by its failure kind

**Goal**: turn a `read_to_string` attempt into a three-word classification.

**Info**: `fs::read_to_string` gives `Ok(String)`, or an `io::Error` whose `kind()` is `NotFound` for a missing file. `std::io` is imported here once for the whole notebook.

**Objectives**

- implement `kind_of_read` returning `"text"`, `"missing"` or `"other"`
- keep the signature unchanged

**See**: §1.1 of the example notebook.

**Hint**: `match` on the `Result`; inside the `Err` arm, match on `e.kind()` against `io::ErrorKind::NotFound` with a `_` catch-all.


In [ ]:
// std::io starts here; std::fs arrived in exercise 0.1.
use std::io;

/// Read `path` as text and classify the attempt: "text", "missing" or "other".
///
/// TODO: replace `todo!()` with a match on the Result and its ErrorKind.
fn kind_of_read(path: &str) -> &'static str {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p").unwrap();
std::fs::write("scratch07p/config.toml", "threads = 4\n").unwrap();
assert_eq!(kind_of_read("scratch07p/config.toml"), "text");
assert_eq!(kind_of_read("scratch07p/definitely-absent.toml"), "missing");
println!("Exercise 1.1 passed");


### Exercise 1.2 — Open a file, a missing file, a directory

**Goal**: classify `File::open` results, including the Windows directory case.

**Info**: on Windows, opening a directory gives `PermissionDenied` ("Access is denied", os error 5); a missing path gives `NotFound`. (On Unix the directory case differs — this course is Windows-verified.)

**Objectives**

- implement `open_report` returning `"opened"`, `"not-found"`, `"denied"` or `"other"`
- keep the signature unchanged

**See**: §1.2 of the example notebook.

**Hint**: same shape as 1.1, but match `io::ErrorKind::PermissionDenied` too; `File::open` is `std::fs::File::open`.


In [ ]:
/// Open `path` and classify the attempt: "opened", "not-found", "denied" or "other".
///
/// TODO: replace `todo!()` with a match on the Result and its ErrorKind.
fn open_report(path: &str) -> &'static str {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p/adir").unwrap();
std::fs::write("scratch07p/plain.txt", "x").unwrap();
assert_eq!(open_report("scratch07p/plain.txt"), "opened");
assert_eq!(open_report("scratch07p/ghost.txt"), "not-found");
assert_eq!(open_report("scratch07p/adir"), "denied");
println!("Exercise 1.2 passed");


### Exercise 1.3 — Add context with `map_err`

**Goal**: re-read §1.3's `load_config`: failures must carry the path *and* the OS reason.

**Objectives**

- implement `load_config` returning the file text on success
- on failure, return `"{path}: could not load config ({e})"`
- keep the signature unchanged

**See**: §1.3 of the example notebook.

**Hint**: `read_to_string(..).map_err(|e| format!(..))` — the closure receives the `io::Error`; `{e}` is its `Display` form.


In [ ]:
/// Load a config file, reporting failures as a self-contained message:
/// `"{path}: could not load config ({os error})"`.
///
/// TODO: replace `todo!()` with read_to_string plus map_err for the context.
fn load_config(path: &str) -> Result<String, String> {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p").unwrap();
std::fs::write("scratch07p/good.toml", "rate = 40\n").unwrap();
assert_eq!(load_config("scratch07p/good.toml"), Ok("rate = 40\n".to_string()));
assert!(matches!(
    load_config("scratch07p/nope.toml"),
    Err(msg) if msg.starts_with("scratch07p/nope.toml: could not load config")
));
println!("Exercise 1.3 passed");


### Exercise 2.1 — Two streams, two strings

**Goal**: produce the data line and the diagnostic line separately.

**Info**: the example cell sent `price=4125.5` to stdout and a narration line to stderr. Here both come back as `String`s so they can be checked; which stream each *belongs* on is the point of the exercise.

**Objectives**

- implement `tick_lines` returning `(data_line, diagnostic_line)`
- the data line must be `price={price}`; the diagnostic must be exactly `recomputed price at tick end`
- keep the signature unchanged

**See**: §2.1 of the example notebook.

**Hint**: `format!` builds the data line; the diagnostic is a fixed string — `.to_string()` it.


In [ ]:
/// Build the two lines a tick produces: the data line and the diagnostic line.
///
/// TODO: replace `todo!()` with two `format!`/`to_string` calls.
fn tick_lines(price: f64) -> (String, String) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(
    tick_lines(4125.5),
    ("price=4125.5".to_string(), "recomputed price at tick end".to_string())
);
assert_eq!(tick_lines(0.0).0, "price=0");
println!("Exercise 2.1 passed");


### Exercise 2.2 — `dbg!` in the middle of an expression

**Goal**: reproduce §2.2's pipeline: scale by 1.08, observe with `dbg!`, round.

**Info**: `dbg!` prints to stderr **and returns its input**, so it can sit inside an expression without changing the value that flows on.

**Objectives**

- implement `dbg_rounded` exactly as the expression `dbg!(raw * 1.08).round()`
- keep the signature unchanged

**See**: §2.2 of the example notebook.

**Hint**: the whole point is that `dbg!(..)` *is* the value — no temporary variable needed.


In [ ]:
/// Scale `raw` by 1.08 (emitting a dbg! line to stderr on the way) and round.
///
/// TODO: replace `todo!()` with the dbg!-wrapped expression.
fn dbg_rounded(raw: f64) -> f64 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(dbg_rounded(4125.5), 4456.0);
assert_eq!(dbg_rounded(100.0), 108.0);
println!("Exercise 2.2 passed");


### Exercise 2.3 — Captured or terminal?

**Goal**: report whether stdout is captured, as the example cell's inverse.

**Info**: under Jupyter — and under this check harness — stdout is captured, so `is_terminal()` is `false` and its negation is `true`. Run the same function from a bare terminal and it flips; that environment-dependence is the lesson.

**Objectives**

- implement `stdout_is_captured` returning the negation of `stdout().is_terminal()`
- keep the signature unchanged

**See**: §2.3 of the example notebook.

**Hint**: `std::io::stdout().is_terminal()` needs the `IsTerminal` trait in scope: `use std::io::IsTerminal;` *inside* your function body is allowed and keeps the notebook-wide import list untouched.


In [ ]:
/// True when stdout is captured (a notebook, a pipe, a file), false on a bare terminal.
///
/// TODO: replace `todo!()` with the negated is_terminal check.
fn stdout_is_captured() -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(stdout_is_captured(), "stdout should be captured under Jupyter and the harness");
println!("Exercise 2.3 passed");


### Exercise 3.1 — Stem and extension

**Goal**: extract the `(stem, extension)` pair the way §3.1's table defines them.

**Info**: `file_stem` strips only the *last* extension: `archive.tar.gz` → stem `archive.tar`, extension `gz`.

**Objectives**

- implement `split_name` returning `(stem, extension)` as owned `String`s
- keep the signature unchanged

**See**: §3.1 of the example notebook.

**Hint**: `std::path::Path::new(p)` then `.file_stem()` / `.extension()` — both return `Option<&OsStr>`; `.unwrap_or_default()` covers both the present and the absent extension, and `.to_string_lossy().into_owned()` makes them `String`.


In [ ]:
/// Split a filename into `(stem, extension)`; `archive.tar.gz` gives `("archive.tar", "gz")`.
///
/// TODO: replace `todo!()` with the Path decomposition.
fn split_name(p: &str) -> (String, String) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(split_name("archive.tar.gz"), ("archive.tar".to_string(), "gz".to_string()));
assert_eq!(split_name("2026-09-25.log"), ("2026-09-25".to_string(), "log".to_string()));
assert_eq!(split_name("README"), ("README".to_string(), "".to_string()));
println!("Exercise 3.1 passed");


### Exercise 3.2 — Build a path, then rewrite its suffix

**Goal**: reproduce §3.2's builder: join a stem, then `set_extension("log")`.

**Info**: `join`/`push` append components (normalising separators to `\` on Windows); `set_extension` replaces the suffix in place.

**Objectives**

- implement `build_log` returning `base/stem.log` as a `PathBuf`
- keep the signature unchanged

**See**: §3.2 of the example notebook.

**Hint**: `std::path::PathBuf::from(base).join(stem)` then `.set_extension("log")` — `set_extension` returns `bool` and mutates in place, so the function's tail expression is the buffer itself.


In [ ]:
/// Build `<base>/<stem>.log`, e.g. `build_log("scratch07p", "run")` ends in `run.log`.
///
/// TODO: replace `todo!()` with join plus set_extension.
fn build_log(base: &str, stem: &str) -> std::path::PathBuf {
    todo!()
}

// --- check (do not modify) ---
let p = build_log("scratch07p", "run");
assert!(p.ends_with("run.log"));
assert!(p.starts_with("scratch07p"));
// joining an absolute path replaces the base entirely:
assert!(std::path::PathBuf::from("relative").join("C:/abs").is_absolute());
println!("Exercise 3.2 passed");


### Exercise 3.3 — Exists, without the error

**Goal**: wrap `fs::exists` into a plain `bool`.

**Info**: `fs::exists` (stable since 1.83) answers the existence question without erroring on a missing file — the error case is reserved for things like permission failures on the *parent* directory.

**Objectives**

- implement `exists_report` unwrapping the `io::Result<bool>`
- keep the signature unchanged

**See**: §3.3 of the example notebook.

**Hint**: one line: `std::fs::exists(path).unwrap()`.


In [ ]:
/// Report whether `path` exists (no error on absence).
///
/// TODO: replace `todo!()` with the fs::exists call.
fn exists_report(path: &str) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(exists_report("scratch07p"), true);
assert_eq!(exists_report("scratch07p/ghost.txt"), false);
println!("Exercise 3.3 passed");


### Exercise 4.1 — The whole-file round trip

**Goal**: write text with `fs::write` and read it back with `fs::read_to_string`.

**Objectives**

- implement `write_then_read` so the returned `String` equals what was written
- keep the signature unchanged

**See**: §4.1 of the example notebook.

**Hint**: two one-shot calls, both `unwrap`-ed; the function's tail expression is the read.


In [ ]:
/// Write `contents` to `path`, then read the file back and return it.
///
/// TODO: replace `todo!()` with the write/read pair.
fn write_then_read(path: &str, contents: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(write_then_read("scratch07p/pos.csv", "sym,qty\nAAPL,300\n"), "sym,qty\nAAPL,300\n");
assert_eq!(write_then_read("scratch07p/pos.csv", "sym,qty\n"), "sym,qty\n"); // overwrites
println!("Exercise 4.1 passed");


### Exercise 4.2 — Text or not text

**Goal**: classify a read as `Ok` text, a missing file, or invalid UTF-8.

**Info**: `read_to_string` gives `ErrorKind::InvalidData` ("stream did not contain valid UTF-8") for non-UTF-8 bytes and `NotFound` for absence. `fs::read` never complains — bytes are bytes.

**Objectives**

- implement `read_text` mapping the error kind to `"not-found"`, `"invalid-data"` or `"other"`
- keep the signature unchanged

**See**: §4.2 of the example notebook.

**Hint**: mirror 1.1's match, adding the `InvalidData` arm; the check writes a `[0xFF, 0xFE]` fixture to trigger it.


In [ ]:
/// Read `path` as text; map failures to "not-found", "invalid-data" or "other".
///
/// TODO: replace `todo!()` with the kind-mapping match.
fn read_text(path: &str) -> Result<String, String> {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p").unwrap();
std::fs::write("scratch07p/utf8.txt", "héllo\n").unwrap();
std::fs::write("scratch07p/utf16.bin", [0xFFu8, 0xFE]).unwrap();
assert_eq!(read_text("scratch07p/utf8.txt"), Ok("héllo\n".to_string()));
assert_eq!(read_text("scratch07p/utf16.bin"), Err("invalid-data".to_string()));
assert_eq!(read_text("scratch07p/ghost.bin"), Err("not-found".to_string()));
println!("Exercise 4.2 passed");


### Exercise 4.3 — Replace a file the atomic-ish way

**Goal**: implement §4.3's pattern: write to a `.tmp` sibling, then `rename` into place.

**Objectives**

- implement `atomic_replace` so `path` ends up with `contents` and no `.tmp` remains
- return the `io::Result<()>` the rename gives you
- keep the signature unchanged

**See**: §4.3 of the example notebook.

**Hint**: build the temp path with `format!("{path}.tmp")`; `fs::write` it, then `fs::rename(tmp, path)` with `?` after the write's `?`.


In [ ]:
/// Replace `path` with `contents` via a `.tmp` sibling and a rename.
///
/// TODO: replace `todo!()` with write-tmp-then-rename, propagating errors with ?.
fn atomic_replace(path: &str, contents: &str) -> std::io::Result<()> {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p").unwrap();
std::fs::write("scratch07p/snap.csv", "old day").unwrap();
atomic_replace("scratch07p/snap.csv", "day close snapshot").unwrap();
assert_eq!(std::fs::read_to_string("scratch07p/snap.csv").unwrap(), "day close snapshot");
assert_eq!(std::fs::exists("scratch07p/snap.csv.tmp").unwrap(), false);
println!("Exercise 4.3 passed");


### Exercise 5.1 — `File::create` truncates

**Goal**: prove the truncate: a long file rewritten through `File::create` leaves only the new bytes.

**Info**: `File::create` = create-or-truncate. `write_all` needs the `Write` trait in scope — imported here once for the rest of the notebook.

**Objectives**

- implement `truncate_write`: create, write, return the file's byte length
- keep the signature unchanged

**See**: §5.1 of the example notebook.

**Hint**: `std::fs::File::create(path)` then `write_all(contents.as_bytes())`, and the length comes from `std::fs::metadata(path).unwrap().len()` (`u64` — cast or keep, the signature says `usize`).


In [ ]:
// the writer trait, first needed here
use std::io::Write;

/// Create-or-truncate `path` with `contents`; return the resulting byte length.
///
/// TODO: replace `todo!()` with create, write_all, and the length.
fn truncate_write(path: &str, contents: &str) -> usize {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p").unwrap();
std::fs::write("scratch07p/trunc.txt", "long existing content that must disappear").unwrap();
assert_eq!(truncate_write("scratch07p/trunc.txt", "0123456789"), 10);
assert_eq!(truncate_write("scratch07p/trunc.txt", "abc"), 3);
assert_eq!(std::fs::read_to_string("scratch07p/trunc.txt").unwrap(), "abc");
println!("Exercise 5.1 passed");


### Exercise 5.2 — Append survives close-and-reopen

**Goal**: add a line in append mode and report the file's total length.

**Info**: append mode positions every write at the end *of that session*; the position survives dropping the handle and reopening — that is why logs work.

**Objectives**

- implement `append_line`: open with `append(true).create(true)`, `writeln!` the line, return the new total length
- keep the signature unchanged

**See**: §5.2 of the example notebook.

**Hint**: `std::fs::OpenOptions::new().append(true).create(true).open(path)`; `writeln!(handle, "{line}")` needs the `Write` trait from 5.1 — already in scope.


In [ ]:
/// Append `line` (with newline) to `path`; return the file's total byte length.
///
/// TODO: replace `todo!()` with the OpenOptions append ritual.
fn append_line(path: &str, line: &str) -> u64 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(append_line("scratch07p/trades.log", "session 1: buy 100 AAPL @ 4125.5"), 33);
assert_eq!(append_line("scratch07p/trades.log", "session 2: sell 100 AAPL @ 4131.0"), 67);
assert_eq!(
    std::fs::read_to_string("scratch07p/trades.log").unwrap(),
    "session 1: buy 100 AAPL @ 4125.5\nsession 2: sell 100 AAPL @ 4131.0\n"
);
println!("Exercise 5.2 passed");


### Exercise 5.3 — `write!` versus `writeln!`

**Goal**: reproduce §5.3's file exactly: a `write!` fragment continued by two `writeln!` lines.

**Info**: `write!` emits no newline; `writeln!` does. Both return the `io::Result` you must unwrap. The example cell built `MSFT 231.75\nAAPL 4125.5\n` from one `write!` and two `writeln!`s.

**Objectives**

- implement `write_quotes` writing that exact content, returning the file's text
- keep the signature unchanged

**See**: §5.3 of the example notebook.

**Hint**: `write!(f, "MSFT ").unwrap();` then `writeln!(f, "{}", b)` then `writeln!(f, "AAPL {}", a)`; the tail expression reads the file back.


In [ ]:
/// Write `MSFT {b}\nAAPL {a}\n` to `path` (one write! fragment, two writeln! lines)
/// and return the file's contents.
///
/// TODO: replace `todo!()` with the three macro calls and the read-back.
fn write_quotes(path: &str, a: f64, b: f64) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(write_quotes("scratch07p/quotes.txt", 4125.5, 231.75), "MSFT 231.75\nAAPL 4125.5\n");
assert_eq!(write_quotes("scratch07p/quotes.txt", 1.0, 2.0), "MSFT 2\nAAPL 1\n");
println!("Exercise 5.3 passed");


### Exercise 6.1 — Small writes wait for the flush

**Goal**: reproduce §6.1: two writes into a 4-byte `BufWriter` stay in memory until `flush()`.

**Info**: buffered writes hold back data until flush/drop/capacity. The check inspects the file *between* the writes and the flush — your body must not flush early.

**Objectives**

- implement `hold_then_flush` returning the file contents **before** and **after** the flush
- keep the signature unchanged

**See**: §6.1 of the example notebook.

**Hint**: create the file, `BufWriter::with_capacity(4, f)`, `write!` the first fragment, read the file for the "before" snapshot, `write!` the second, `flush()`, read again. The `Write` trait is in scope from 5.1.


In [ ]:
// the reader traits, first needed here
use std::io::{BufRead, BufReader};

/// Write "ab" then "cd" through a 4-byte BufWriter, capturing the file contents
/// before the flush and after it: returns (before, after).
///
/// TODO: replace `todo!()` with the buffered writes and the two snapshots.
fn hold_then_flush(path: &str) -> (String, String) {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p").unwrap();
let (before, after) = hold_then_flush("scratch07p/buf.txt");
assert_eq!(before, "");
assert_eq!(after, "abcd");
println!("Exercise 6.1 passed");


### Exercise 6.2 — Line access: keep it or strip it

**Goal**: read one line with `read_line` (newline kept) and the rest with `lines()` (stripped).

**Info**: §6.2's fixture ends without a trailing newline and `lines()` still yields the final partial line.

**Objectives**

- implement `split_lines` returning `(first_line_with_newline, remaining_lines)`
- keep the signature unchanged

**See**: §6.2 of the example notebook.

**Hint**: `BufReader::new(std::fs::File::open(path).unwrap())`, `read_line(&mut String::new())` for the first, then `.lines().map(|l| l.unwrap()).collect()` for the rest.


In [ ]:
/// Read the first line with its newline, then the rest with lines() (stripped).
///
/// TODO: replace `todo!()` with read_line plus the lines() iterator.
fn split_lines(path: &str) -> (String, Vec<String>) {
    todo!()
}

// --- check (do not modify) ---
std::fs::write("scratch07p/feed.txt", "AAPL 4125.5\nMSFT 231.75\nNVDA 187.2").unwrap();
assert_eq!(split_lines("scratch07p/feed.txt"), (
    "AAPL 4125.5\n".to_string(),
    vec!["MSFT 231.75".to_string(), "NVDA 187.2".to_string()]
));
println!("Exercise 6.2 passed");


### Exercise 6.3 — Lines from memory, no file

**Goal**: run the same line-splitting against a `&[u8]`, the way §6.3 dropped the filesystem.

**Info**: `&[u8]` implements `Read` and `BufRead` directly, so `read_line` and `lines()` work on in-memory bytes.

**Objectives**

- implement `slice_lines` splitting a byte slice into its lines (stripped, owned)
- keep the signature unchanged

**See**: §6.3 of the example notebook.

**Hint**: `data.lines().map(|l| l.unwrap()).collect()` — the trait is in scope from 6.1; nothing else is needed.


In [ ]:
/// Split a byte slice into its lines (newlines stripped, owned strings).
///
/// TODO: replace `todo!()` with the lines() iterator over the slice.
fn slice_lines(data: &[u8]) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(slice_lines(b"one\ntwo\nthree\n"), vec!["one", "two", "three"]);
assert_eq!(slice_lines(b""), Vec::<String>::new());
assert_eq!(slice_lines(b"no newline at end"), vec!["no newline at end"]);
println!("Exercise 6.3 passed");


### Exercise 7.1 — List a directory, sorted

**Goal**: return `scratch07p/quotes`'s file names in sorted order, the §7.1 way.

**Info**: `read_dir` order is OS-dependent — always sort before showing anything.

**Objectives**

- implement `sorted_entries` listing the *file names* of a directory, sorted
- keep the signature unchanged

**See**: §7.1 of the example notebook.

**Hint**: `std::fs::read_dir(dir).unwrap()` → `.map(|e| e.unwrap().file_name().to_string_lossy().into_owned())` → `collect` into a `Vec`, then `.sort()` it.


In [ ]:
/// List the entries of `dir` as sorted owned names.
///
/// TODO: replace `todo!()` with read_dir plus the name mapping and a sort.
fn sorted_entries(dir: &str) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p/quotes").unwrap();
std::fs::write("scratch07p/quotes/msft.csv", "").unwrap();
std::fs::write("scratch07p/quotes/aapl.csv", "").unwrap();
std::fs::write("scratch07p/quotes/readme.md", "").unwrap();
assert_eq!(
    sorted_entries("scratch07p/quotes"),
    vec!["aapl.csv", "msft.csv", "readme.md"]
);
println!("Exercise 7.1 passed");


### Exercise 7.2 — Metadata without opening

**Goal**: classify each entry of a directory by kind and size, as §7.2's table did.

**Objectives**

- implement `entry_report` returning `(kind, size)` per entry, sorted by name
- `kind` is `"dir "` (padded, as the example printed it) or `"file"`
- keep the signature unchanged

**See**: §7.2 of the example notebook.

**Hint**: `entry.metadata().unwrap()` gives `.len()` (bytes, `u64`) and `.is_dir()`; build the `(String, u64)` rows for *files only inside the listing* — directories still appear, just with their own kind.


In [ ]:
/// Report every entry of `dir` as (kind, size-in-bytes), sorted by name.
/// The kind is "dir " for directories and "file" for files, like the example's table.
///
/// TODO: replace `todo!()` with read_dir, metadata, and the row building.
fn entry_report(dir: &str) -> Vec<(String, u64)> {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p/meta").unwrap();
std::fs::write("scratch07p/meta/a.txt", "12345").unwrap();
std::fs::create_dir_all("scratch07p/meta/sub").unwrap();
std::fs::write("scratch07p/meta/b.bin", [0u8; 10]).unwrap();
let report = entry_report("scratch07p/meta");
assert_eq!(report.len(), 3);
assert!(report.contains(&("file".to_string(), 5)));
assert!(report.contains(&("file".to_string(), 10)));
assert!(report.iter().any(|(kind, _)| kind == "dir "));
println!("Exercise 7.2 passed");


### Exercise 7.3 — The propagating walk

**Goal**: implement §7.3's recursive `tree_size` with the `?` strategy: one unreadable entry aborts.

**Objectives**

- implement `tree_size` summing byte sizes over a directory tree, recursively
- return `io::Result<u64>`; propagate every failure with `?`
- keep the signature unchanged

**See**: §7.3 of the example notebook.

**Hint**: for each entry: `entry?`, `entry.metadata()?`, then recurse on directories via `entry.path()` or add `md.len()`; the function's tail is `Ok(total)`.


In [ ]:
/// Total byte size of the tree rooted at `dir`; the first unreadable entry aborts.
///
/// TODO: replace `todo!()` with the recursive walk, propagating with ?.
fn tree_size(dir: &std::path::Path) -> std::io::Result<u64> {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p/tree/deep/deeper").unwrap();
std::fs::write("scratch07p/tree/a.txt", [0u8; 300]).unwrap();
std::fs::write("scratch07p/tree/deep/b.txt", [0u8; 500]).unwrap();
std::fs::write("scratch07p/tree/deep/deeper/c.txt", [0u8; 200]).unwrap();
assert_eq!(tree_size(std::path::Path::new("scratch07p/tree")).unwrap(), 1000);
assert!(tree_size(std::path::Path::new("scratch07p/ghost-tree")).is_err());
println!("Exercise 7.3 passed");


### Exercise 7.4 — The lenient walk

**Goal**: the §7.4 variant: skip what cannot be read, return a plain number.

**Objectives**

- implement `tree_size_lenient`: same total, but unreadable entries contribute zero
- a missing root yields `0`, not an error
- keep the signature unchanged

**See**: §7.4 of the example notebook.

**Hint**: `if let Ok(entries) = fs::read_dir(dir)` guards the root; inside, `filter_map(|e| e.ok())` and `if let Ok(md) = ...` do the skipping; recurse on directories.


In [ ]:
/// Best-effort byte size of the tree at `dir`; unreadable subtrees contribute zero.
///
/// TODO: replace `todo!()` with the skip-and-continue walk.
fn tree_size_lenient(dir: &std::path::Path) -> u64 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(tree_size_lenient(std::path::Path::new("scratch07p/tree")), 1000);
assert_eq!(tree_size_lenient(std::path::Path::new("scratch07p/ghost-tree")), 0);
println!("Exercise 7.4 passed");


### Exercise 8.1 — The tail after the program name

**Goal**: element 0 of an argv vector is the program name; return everything after it.

**Info**: under the kernel, `env::args()` belongs to the host, so this exercise works on an argv vector passed in — the same shape a real `main` would receive.

**Objectives**

- implement `argv_tail` skipping the first element and collecting the rest
- keep the signature unchanged

**See**: §8.1 of the example notebook.

**Hint**: `argv.into_iter().skip(1).collect()` — `into_iter` because the vector is yours.


In [ ]:
/// Drop the program name (element 0) and collect the remaining arguments.
///
/// TODO: replace `todo!()` with the skip-and-collect.
fn argv_tail(argv: Vec<String>) -> Vec<String> {
    todo!()
}

// --- check (do not modify) ---
let args: Vec<String> = ["feed.exe", "--fast", "AAPL"].iter().map(|s| s.to_string()).collect();
assert_eq!(argv_tail(args), vec!["--fast", "AAPL"]);
assert_eq!(argv_tail(vec!["lone.exe".to_string()]), Vec::<String>::new());
assert_eq!(argv_tail(Vec::new()), Vec::<String>::new());
println!("Exercise 8.1 passed");


### Exercise 8.2 — A safe index into argv

**Goal**: the `nth(n)` contract: out-of-range is `None`, never a panic.

**Objectives**

- implement `count_args` returning `Some(count)` when index `n` exists, `None` otherwise
- keep the signature unchanged

**See**: §8.1 of the example notebook.

**Hint**: `argv.get(n)` is the slice analogue of `nth`; `.is_some()` turns it into the answer.


In [ ]:
/// `Some(number of arguments)` when index `n` exists in `argv`, otherwise `None`.
///
/// TODO: replace `todo!()` with the bounds-safe lookup.
fn count_args(argv: &[String], n: usize) -> Option<usize> {
    todo!()
}

// --- check (do not modify) ---
let args: Vec<String> = ["p", "a", "b"].iter().map(|s| s.to_string()).collect();
assert_eq!(count_args(&args, 0), Some(3));
assert_eq!(count_args(&args, 2), Some(3));
assert_eq!(count_args(&args, 3), None);
assert_eq!(count_args(&[], 0), None);
println!("Exercise 8.2 passed");


### Exercise 9.1 — Read an env var without unwrap

**Goal**: turn `env::var` into a three-way classification, §9.1 style.

**Info**: `VarError::NotPresent` for absence, `VarError::NotUnicode` for non-UTF-8 bytes; `Ok` means usable text.

**Objectives**

- implement `fetch_env` returning `"set: {value}"`, `"absent"` or `"non-unicode"`
- keep the signature unchanged

**See**: §9.1 of the example notebook.

**Hint**: match on `std::env::var(key)`; the `Ok` arm formats `set: {v}`; `NotPresent` and `NotUnicode(_)` are the two `Err` shapes.


In [ ]:
/// Classify `key`: "set: {value}", "absent", or "non-unicode".
///
/// TODO: replace `todo!()` with the three-way match.
fn fetch_env(key: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
unsafe { std::env::set_var("NB07P_KNOWN", "40") };
assert_eq!(fetch_env("NB07P_KNOWN"), "set: 40");
assert_eq!(fetch_env("NB07P_DEFINITELY_ABSENT"), "absent");
println!("Exercise 9.1 passed");


### Exercise 9.2 — Set, read, remove

**Goal**: the §9.2 lifecycle around one `unsafe`-guarded pair of calls.

**Objectives**

- implement `set_then_remove`: set `key` to `value`, read it back, remove it, read again
- return `(read_after_set, read_after_remove)` where the second read reports absence as `"absent"`
- keep the signature unchanged

**See**: §9.2 of the example notebook.

**Hint**: `unsafe { std::env::set_var(key, value) }`, `fetch_env(key)` twice — the second after `unsafe { std::env::remove_var(key) }`. Reusing `fetch_env` keeps the classification in one place.


In [ ]:
/// Set `key` to `value`, read it, remove it, read again: (after_set, after_remove).
///
/// TODO: replace `todo!()` with the set/read/remove/read ritual.
fn set_then_remove(key: &str, value: &str) -> (String, String) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(
    set_then_remove("NB07P_CYCLE", "on"),
    ("set: on".to_string(), "absent".to_string())
);
println!("Exercise 9.2 passed");


### Exercise 10.1 — Capture a program's output

**Goal**: wrap §10.1's `git --version` capture: success flag plus decoded stdout.

**Info**: output comes back as bytes; `String::from_utf8_lossy` decodes. A program that runs but fails is still `Ok` — only a missing program is `Err`.

**Objectives**

- implement `command_report` returning `(success, trimmed_stdout)` for `git --version`
- keep the signature unchanged

**See**: §10.1 of the example notebook.

**Hint**: `std::process::Command::new("git").arg("--version").output().unwrap()`; the two fields are `out.status.success()` and `String::from_utf8_lossy(&out.stdout).trim().to_string()`.


In [ ]:
/// Run `git --version`; return (success, trimmed stdout).
///
/// TODO: replace `todo!()` with the Command capture and decode.
fn command_report() -> (bool, String) {
    todo!()
}

// --- check (do not modify) ---
let (ok, version) = command_report();
assert!(ok);
assert!(version.starts_with("git version"), "got: {version}");
println!("Exercise 10.1 passed");


### Exercise 10.2 — Exit codes are data

**Goal**: run a command that *fails on purpose* and hand back its exit code.

**Info**: `cmd /C exit 3` (Windows; `sh -c "exit 3"` on Unix) exits with code 3: an `Ok` output carrying non-success. `status.code()` is `Option<i32>`.

**Objectives**

- implement `exit_code_of` returning `Some(code)` when the child ran, `None` when it could not be started
- keep the signature unchanged

**See**: §10.2 of the example notebook.

**Hint**: match on `Command::new(program).args(args).status()`: `Ok(st) => st.code()`, `Err(_) => None`.


In [ ]:
/// Run `program` with `args`; Some(exit code) when it ran, None when it could not start.
///
/// TODO: replace `todo!()` with the status match.
fn exit_code_of(program: &str, args: &[&str]) -> Option<i32> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(exit_code_of("cmd", &["/C", "exit 3"]), Some(3)); // Unix: sh -c "exit 3"
assert_eq!(exit_code_of("definitely-not-a-program-07p", &[]), None);
println!("Exercise 10.2 passed");


### Exercise 11.1 — Read a panic's payload

**Goal**: catch the panic from a failed `expect` and return its message, §11.1 style.

**Info**: the payload is `String` or `&str` after downcasting; `expect("msg")` prefixes the error's `Debug` form with your message.

**Objectives**

- implement `panic_message`: run `f` inside `catch_unwind`, returning `Ok(value)` or `Err(message)`
- keep the signature unchanged

**See**: §11.1 of the example notebook.

**Hint**: `std::panic::catch_unwind(f)` gives a `Result<T, Box<dyn Any + Send>>`; on `Err`, downcast to `String` first, then `&str`, with `unwrap_or_default()` as the floor.


In [ ]:
/// Run `f`, catching a panic: Ok(value) normally, Err(message) when it panicked.
///
/// TODO: replace `todo!()` with catch_unwind plus the payload downcasts.
fn panic_message<T>(f: fn() -> T) -> Result<T, String> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(panic_message(|| 41i32), Ok(41));
let msg = panic_message(|| panic!("boom {}", 7)).unwrap_err();
assert!(msg.contains("boom 7"), "got: {msg}");
println!("Exercise 11.1 passed");


### Exercise 11.2 — The four E-codes of this chapter

**Goal**: the §11 catalogue as a lookup: given the trigger, name the code.

**Objectives**

- implement `error_code` mapping the four triggers to their codes
- keep the signature unchanged

**See**: §11 of the example notebook.

**Hint**: the triggers in order: printing a `Result` with `{}`; an `Option` where a `Result` is expected; two live `&mut` writers on one file; a trait method used without the trait in scope.


In [ ]:
/// Map a diagnostic trigger to its error code:
/// 0 = printing a Result with {}, 1 = Option where Result is expected,
/// 2 = two live &mut writers, 3 = trait method without the trait in scope.
///
/// TODO: replace `todo!()` with the four-arm match.
fn error_code(trigger: u8) -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(error_code(0), "E0277");
assert_eq!(error_code(1), "E0308");
assert_eq!(error_code(2), "E0499");
assert_eq!(error_code(3), "E0599");
assert_eq!(error_code(9), "unknown");
println!("Exercise 11.2 passed");


### Exercise 12.1 — Measure the buffering win

**Goal**: the §12.1 experiment as data: write the same 2000 lines both ways and report the ratio.

**Info**: the ratio is machine-dependent (30–50× here); the *structural* facts — identical bytes, a positive ratio — are what the check can rely on anywhere.

**Objectives**

- implement `buffered_ratio` writing `lines` records through a bare `File` and through a `BufWriter` (flush at the end), timing both
- return `(files_identical, ratio_of_unbuffered_to_buffered)`
- keep the signature unchanged

**See**: §12.1 of the example notebook.

**Hint**: two `{ .. }` blocks with `Instant::now()`/`elapsed()` around each; `BufWriter::new` + a final `flush().unwrap()`; the ratio is `unbuf.as_secs_f64() / bufd.as_secs_f64()`.


In [ ]:
// the timer, first needed here
use std::time::Instant;

/// Write `lines` "tick-N" records to `a_path` unbuffered and `b_path` buffered;
/// return (contents identical, unbuffered-time / buffered-time).
///
/// TODO: replace `todo!()` with the two timed writes and the comparison.
fn buffered_ratio(a_path: &str, b_path: &str, lines: u32) -> (bool, f64) {
    todo!()
}

// --- check (do not modify) ---
let (identical, ratio) = buffered_ratio("scratch07p/unbuf.log", "scratch07p/bufd.log", 2000);
assert!(identical, "both writers must produce the same bytes");
assert!(ratio > 0.0 && ratio.is_finite());
println!("Exercise 12.1 passed (ratio on this machine: {ratio:.1}x)");


### Exercise 13.1 — Parse `key=value` lines through one `io::Result`

**Goal**: §13.1's `load_pairs`: whole-file read, per-line parse, two failure kinds, one signature.

**Objectives**

- implement `load_pairs` returning the `(key, value)` pairs, both trimmed
- a line without `=` is `ErrorKind::InvalidData`; a missing file surfaces as its own kind
- keep the signature unchanged

**See**: §13.1 of the example notebook.

**Hint**: `fs::read_to_string(path)?` for the file; inside the loop, `line.split_once('=').ok_or_else(|| std::io::Error::new(std::io::ErrorKind::InvalidData, format!("bad line: {line:?}")))?`.


In [ ]:
/// Read `key=value` lines from `path`; every failure path is visible in the signature.
///
/// TODO: replace `todo!()` with the read plus per-line split_once parsing.
fn load_pairs(path: &str) -> std::io::Result<Vec<(String, String)>> {
    todo!()
}

// --- check (do not modify) ---
std::fs::write("scratch07p/pairs.txt", "host=ny4\nrate= 40\n").unwrap();
assert_eq!(
    load_pairs("scratch07p/pairs.txt").unwrap(),
    vec![("host".to_string(), "ny4".to_string()), ("rate".to_string(), "40".to_string())]
);
assert_eq!(load_pairs("scratch07p/nope.cfg").unwrap_err().kind(), std::io::ErrorKind::NotFound);
std::fs::write("scratch07p/broken.cfg", "host=ny4\nno equals sign\n").unwrap();
assert_eq!(load_pairs("scratch07p/broken.cfg").unwrap_err().kind(), std::io::ErrorKind::InvalidData);
println!("Exercise 13.1 passed");


## 16. Review: Mixed Problems

Two problems that chain several sections together — no new material, only assembly.

### Exercise 16.1 — Ingest a price feed from memory

**Goal**: parse `"SYM price"` lines *from a byte slice* into `(symbol, price)` pairs.

**Info**: everything used here is from §6 (line access on a `&[u8]`) plus chapter 03's parsing; no file is touched. `read_line` **appends** into the buffer you hand it — clear it between lines.

**Objectives**

- implement `ingest_quotes` returning the parsed pairs in order
- keep the signature unchanged

**See**: §6.3 and §13.1 of the example notebook.

**Hint**: loop while `text.as_bytes().read_line(&mut line).unwrap() > 0`; inside, `line.trim().split_once(' ')` and `price.parse().unwrap()`; `line.clear()` before the next round.


In [ ]:
/// Parse "SYM price" lines from an in-memory byte slice into owned pairs.
///
/// TODO: replace `todo!()` with the read_line loop over text.as_bytes().
fn ingest_quotes(text: &str) -> Vec<(String, f64)> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(
    ingest_quotes("AAPL 4125.5\nMSFT 231.75\nNVDA 187.2\n"),
    vec![("AAPL".to_string(), 4125.5), ("MSFT".to_string(), 231.75), ("NVDA".to_string(), 187.2)]
);
assert_eq!(ingest_quotes(""), Vec::<(String, f64)>::new());
assert_eq!(ingest_quotes("AAPL 4125.5"), vec![("AAPL".to_string(), 4125.5)]);
println!("Exercise 16.1 passed");


### Exercise 16.2 — Strike the sandbox

**Goal**: the chapter's last act: delete a whole tree and confirm it is gone.

**Info**: `remove_dir_all` removes a directory and everything inside it; it errors (`NotFound`) when the target is already gone — which is exactly what makes the second check meaningful.

**Objectives**

- implement `sandbox_wipe` returning `true` when the tree existed and was removed
- keep the signature unchanged

**See**: §7 and §12.2 of the example notebook.

**Hint**: `std::fs::remove_dir_all(path).is_ok()` is the whole body; the check wipes every fixture the notebook built, so run this cell last.


In [ ]:
/// Remove the tree at `path`; true when it existed and is now gone.
///
/// TODO: replace `todo!()` with the remove_dir_all call.
fn sandbox_wipe(path: &str) -> bool {
    todo!()
}

// --- check (do not modify) ---
std::fs::create_dir_all("scratch07p/tree/deep/deeper").unwrap();
std::fs::write("scratch07p/tree/final.txt", "bye").unwrap();
assert_eq!(sandbox_wipe("scratch07p"), true);
assert_eq!(std::fs::exists("scratch07p").unwrap(), false);
assert_eq!(sandbox_wipe("scratch07p"), false); // already gone
println!("Exercise 16.2 passed");


### What Comes Next

Chapter 07's exercises close the core-toolbox arc. From here the notebooks assume you can read, write, and classify failures across the program boundary without ceremony.

| Chapter | Title | What it settles |
|---|---|---|
| **08** | **Assertions and Type Checks** | `assert!`, `matches!`, `size_of` — the checking toolkit this notebook's grader cells quietly used |
| 09 | Logging | `tracing` events and spans — the structured upgrade over this chapter's `eprintln!` diagnostics |
| 11 | Errors in Depth | real error types for the `io::Error`s you matched by kind here |

**Carry forward.** You should now be able to do these without looking anything up:

- read and write files three ways (one-shot, handle, buffered) and pick correctly under pressure
- classify every `io::Error` by kind and know which kinds this platform actually produces
- walk a directory tree twice: once strict with `?`, once lenient with skips
- run another program and treat its exit code as data, not as an exception
- predict where a `flush()` is mandatory and what a dropped `BufWriter` cannot tell you
